In [1]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L
!unzip -q "/content/drive/MyDrive/TCGer-detector/tcger-card-yolo.zip" -d /content/
!ls /content/yolo-dataset && head -20 /content/yolo-dataset/data.yaml

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU 0: NVIDIA L4 (UUID: GPU-5b3da6d1-ffa1-c109-a668-afe86eeeecf3)
data.yaml  images  labels
path: .
train: images/train
val: images/val
test: images/test
names:
  0: card


In [2]:

!pip -q install ultralytics
import yaml
cfg = yaml.safe_load(open('/content/yolo-dataset/data.yaml'))
cfg['path'] = '/content/yolo-dataset'
yaml.safe_dump(cfg, open('/content/yolo-dataset/data.yaml', 'w'))
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
model.train(data='/content/yolo-dataset/data.yaml', epochs=60, imgsz=640, batch=32, project='/content/runs', name='card')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 76.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo-dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_mo

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ad7d5b34080>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [3]:

import yaml
from ultralytics import YOLO
best = YOLO('/content/runs/card/weights/best.pt')
m1 = best.val(data='/content/yolo-dataset/data.yaml', split='test')
cfg = yaml.safe_load(open('/content/yolo-dataset/data.yaml'))
cfg['val'] = 'images/tight_test'
yaml.safe_dump(cfg, open('/content/tight.yaml', 'w'))
m2 = best.val(data='/content/tight.yaml')
print('SCENE-TEST mAP50', m1.box.map50, 'TIGHT-TEST mAP50', m2.box.map50)
p = best.export(format='coreml', nms=True, imgsz=640)
print('exported', p)
!cp -r /content/runs/card/weights/best.mlpackage "/content/drive/MyDrive/TCGer-detector/CardDetector-yolo11s.mlpackage"
!ls -la "/content/drive/MyDrive/TCGer-detector/"

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2247.3±889.6 MB/s, size: 103.3 KB)
val: Scanning /content/yolo-dataset/labels/test... 136 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 136/136 1.5Kit/s 0.1s
val: New cache created: /content/yolo-dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.3it/s 2.1s
                   all        136        166      0.994      0.985      0.995      0.985
Speed: 1.9ms preprocess, 6.0ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to /content/runs/detect/val
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1822.3±1011.4 MB/s, size: 87.6 KB)
val: Scanning /content/yolo-dataset/labels/ti

/usr/local/lib/python3.12/dist-packages/coremltools/optimize/torch/palettization/fake_palettize.py:82: SyntaxWarning: invalid escape sequence '\_'
  n_bits (:obj:`int`): Number of palettization bits. There would be :math:`2^{n\_bits}` unique weights in the ``LUT``.



CoreML: starting export with coremltools 9.0...


Running MIL default pipeline:  11%|█         | 10/95 [00:00<00:00, 92.51 passes/s]/usr/local/lib/python3.12/dist-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '1149', of the source model, has been renamed to 'var_1149' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
/usr/local/lib/python3.12/dist-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '1151', of the source model, has been renamed to 'var_1151' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 85.20 passes/s]


CoreML: starting pipeline with coremltools 9.0...
CoreML: pipeline success
CoreML: export success ✅ 16.5s, saved as '/content/runs/card/weights/best.mlpackage' (18.2 MB)

Export complete (16.8s)
Results saved to /content/runs/card/weights/best.mlpackage
Predict:         yolo predict task=detect model=/content/runs/card/weights/best.mlpackage imgsz=640 
Validate:        yolo val task=detect model=/content/runs/card/weights/best.mlpackage imgsz=640 data=/content/yolo-dataset/data.yaml  
Visualize:       https://netron.app
exported /content/runs/card/weights/best.mlpackage
total 617851
drwx------ 3 root root      4096 Aug  9 18:23 CardDetector-yolo11s.mlpackage
-rw------- 1 root root 632675044 Aug  9 16:47 tcger-card-yolo.zip
